In [1]:
# Importing Libraries
import selenium
import pandas as pd
import time
import os
from bs4 import BeautifulSoup

# Importing selenium webdriver 
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys

# Importing required Exceptions which needs to handled
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException

#Importing requests
import requests

# importing regex
import re

In [3]:
import time
import random
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# ---------------- START DRIVER ----------------

driver = webdriver.Edge()
driver.maximize_window()

wait = WebDriverWait(driver,10)


# ---------------- CITY LIST ----------------

cities = [
    "delhi","mumbai","bangalore","hyderabad","chennai",
    "pune","kolkata","ahmedabad","jaipur","lucknow",
    "chandigarh","indore","nagpur","surat","vadodara"
]


dataset = []


# ---------------- FUNCTION: GET CAR DETAILS ----------------

def get_car_details(url):

    driver.get(url)

    wait.until(
        EC.presence_of_element_located((By.XPATH,"//table"))
    )

    # ----- TITLE -----
    try:
        title = driver.find_element(By.XPATH,"//h1").text
    except:
        title = "-"

    # ----- PRICE -----
    try:
        price = driver.find_element(By.XPATH,"//span[contains(text(),'₹')]").text
    except:
        price = "-"

    # ----- CITY -----
    try:
        city = driver.find_element(By.XPATH,"//td[text()='CITY']/following-sibling::td").text
    except:
        city = "-"

    # ----- FUEL -----
    try:
        fuel = driver.find_element(By.XPATH,"//td[text()='FUEL TYPE']/following-sibling::td").text
    except:
        fuel = "-"

    # ----- KM -----
    try:
        kms = driver.find_element(By.XPATH,"//td[contains(text(),'KMS')]/following-sibling::td").text
    except:
        kms = "-"

    # ----- OWNERS -----
    try:
        owners = driver.find_element(By.XPATH,"//td[contains(text(),'OWNER')]/following-sibling::td").text
    except:
        owners = "-"

    return {
        "title": title,
        "price": price,
        "city": city,
        "fuel": fuel,
        "kms": kms,
        "owners": owners,
        "url": url
    }


# ---------------- SCRAPE LISTING PAGES ----------------

for city in cities:

    print("Scraping city:", city)

    for page in range(1,150):

        listing_url = f"https://www.cartrade.com/second-hand/{city}/?page={page}"

        driver.get(listing_url)

        try:

            wait.until(
                EC.presence_of_all_elements_located((By.XPATH,"//a[contains(@href,'/second-hand/')]"))
            )

        except:
            print("No cars found on page:",page)
            break


        cars = driver.find_elements(By.XPATH,"//a[contains(@href,'/second-hand/')]")

        car_links = []

        cars = driver.find_elements(By.XPATH, "//h3[contains(@class,'car-name')]/ancestor::a")
        
        car_links = []
        
        for car in cars:
        
            link = car.get_attribute("href")
        
            if link:
                car_links.append(link)

        car_links = list(set(car_links))

        print("Cars found:",len(car_links))


        # -------- SCRAPE DETAIL PAGES --------

        for url in car_links:

        if url in scraped_urls:
            continue

            try:

                data = get_car_details(url)

                dataset.append(data)

                # save every 200 cars
                if len(dataset) % 200 == 0:

                    df = pd.DataFrame(dataset)
                    df.to_csv("used_car_dataset.csv",index=False)

                    print("Saved:",len(dataset),"cars")

                time.sleep(random.uniform(1,2))

            except:

                print("Skipping:",url)


# ---------------- FINAL SAVE ----------------

df = pd.DataFrame(dataset)

df.to_csv("used_car_dataset.csv",index=False)

print("Scraping complete:",len(dataset),"cars")


driver.quit()

IndentationError: expected an indented block after 'for' statement on line 133 (2869927898.py, line 135)

In [9]:
df = pd.read_csv("used_car_dataset.csv")
print(len(df))

5721


In [6]:
df.to_excel('df.xlsx')

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Edge()
wait = WebDriverWait(driver,10)

prices = []

for url in df["url"]:

    driver.get(url)

    try:
        price = driver.find_element(By.XPATH,"//span[contains(@class,'amount')]").text
    except:
        price = "-"

    prices.append(price)

    time.sleep(1)

driver.quit()

In [ ]:
df["price"] = df["price"].str.replace("₹","")
df["price"] = df["price"].str.replace(",","")
df["price"] = df["price"].str.replace(" Lakh","")

df["price"] = df["price"].astype(float) * 100000

In [5]:
!pip install requests
!pip install beautifulsoup4

In [6]:
import pandas as pd

df = pd.read_csv("used_car_dataset.csv")

urls = df["url"].tolist()

In [7]:
import requests
from bs4 import BeautifulSoup

prices = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for url in urls:

    try:
        response = requests.get(url, headers=headers)

        soup = BeautifulSoup(response.text, "html.parser")

        price_tag = soup.select_one("span.amount")

        if price_tag:
            price = price_tag.text.strip()
        else:
            price = "-"

    except:
        price = "-"

    prices.append(price)

print("Finished scraping prices")

Finished scraping prices


In [8]:
df["price"] = prices

df.to_csv("used_car_dataset_with_price.csv", index=False)

In [12]:
# convert column to string first
df["price"] = df["price"].astype(str)

# remove symbols
df["price"] = df["price"].str.replace("₹", "", regex=False)
df["price"] = df["price"].str.replace(",", "", regex=False)
df["price"] = df["price"].str.replace(" Lakh", "", regex=False)

# replace missing values
df["price"] = df["price"].replace("-", None)

# convert safely to numeric
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# convert lakh to rupees
df["price"] = df["price"] * 100000

In [14]:
df = df.dropna(subset=["price"])

In [15]:
df["price"].head()
df["price"].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: price, dtype: float64

In [16]:
# convert column to string first
df["price"] = df["price"].astype(str)

# remove symbols
df["price"] = df["price"].str.replace("₹", "", regex=False)
df["price"] = df["price"].str.replace(",", "", regex=False)
df["price"] = df["price"].str.replace(" Lakh", "", regex=False)

# replace missing values
df["price"] = df["price"].replace("-", None)

# convert safely to numeric
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# convert lakh to rupees
df["price"] = df["price"] * 100000

In [18]:
df["price"].dtype

dtype('int64')

In [19]:
df["price"] = df["price"].astype(str)

In [20]:
df["price"] = df["price"].str.replace("₹","", regex=False)
df["price"] = df["price"].str.replace(",","", regex=False)
df["price"] = df["price"].str.replace(" Lakh","", regex=False)

In [21]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["price"] = df["price"] * 100000

In [22]:
df = df.dropna(subset=["price"])

In [23]:
df["price"].head()
df["price"].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: price, dtype: float64

In [24]:
df["price"].unique()[:20]

array([], dtype=int64)

In [25]:
df["price"].value_counts().head(20)

Series([], Name: count, dtype: int64)

In [26]:
import pandas as pd

df = pd.read_csv("used_car_dataset_with_price.csv")

print(len(df))
df.head()

7007


,title,price,city,fuel,kms,owners,url
0,2020 Toyota Fortuner 2.8 4x2 MT,-,Delhi,Diesel,"35,106 Kms",First,https://www.cartrade.com/second-hand/delhi/toy...
1,2021 Tata Nexon EV XZ Plus,-,Delhi,Electric,"52,000 Kms",First,https://www.cartrade.com/second-hand/delhi/tat...
2,2023 Toyota Innova Hycross ZX (O) Hybrid (Elec...,-,Delhi,Hybrid (Electric + Petrol),"56,835 Kms",First,https://www.cartrade.com/second-hand/delhi/toy...
3,2021 Toyota Camry Hybrid,-,Delhi,Hybrid (Electric + Petrol),"63,000 Kms",First,https://www.cartrade.com/second-hand/delhi/toy...
4,2019 Mercedes-Benz C-Class C 200 Progressive [...,-,Delhi,Petrol,"40,500 Kms",First,https://www.cartrade.com/second-hand/delhi/mer...


In [31]:
import requests
from bs4 import BeautifulSoup

prices = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for url in urls:

    try:
        r = requests.get(url, headers=headers)
        soup = BeautifulSoup(r.text, "html.parser")

        price_tag = soup.select_one("span.amount")

        if price_tag:
            price = price_tag.text.strip()
        else:
            price = None

    except:
        price = None

    prices.append(price)

print("Finished scraping prices")

Finished scraping prices


In [4]:
import requests
from bs4 import BeautifulSoup

prices = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for url in urls:

    try:
        r = requests.get(url, headers=headers)

        soup = BeautifulSoup(r.text, "html.parser")

        price_tag = soup.select_one("span[content]")

        if price_tag:
            price = price_tag["content"]
        else:
            price = None

    except:
        price = None

    prices.append(price)

print("Price scraping finished")

Price scraping finished


In [5]:
df["price"] = prices

In [8]:
df.head()

,title,price,city,fuel,kms,owners,url
0,2020 Toyota Fortuner 2.8 4x2 MT,None,Delhi,Diesel,"35,106 Kms",First,https://www.cartrade.com/second-hand/delhi/toy...
1,2021 Tata Nexon EV XZ Plus,860000,Delhi,Electric,"52,000 Kms",First,https://www.cartrade.com/second-hand/delhi/tat...
2,2023 Toyota Innova Hycross ZX (O) Hybrid (Elec...,2950000,Delhi,Hybrid (Electric + Petrol),"56,835 Kms",First,https://www.cartrade.com/second-hand/delhi/toy...
3,2021 Toyota Camry Hybrid,2690000,Delhi,Hybrid (Electric + Petrol),"63,000 Kms",First,https://www.cartrade.com/second-hand/delhi/toy...
4,2019 Mercedes-Benz C-Class C 200 Progressive [...,3149000,Delhi,Petrol,"40,500 Kms",First,https://www.cartrade.com/second-hand/delhi/mer...


In [9]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")

In [10]:
df = df.dropna(subset=["price"])

In [11]:
df.to_csv("car_price_dataset_final.csv", index=False)

In [15]:
df["year"] = df["title"].str.extract(r'(\d{4})').astype(int)

In [16]:
df.head()

,title,price,city,fuel,kms,owners,url,year
1,2021 Tata Nexon EV XZ Plus,860000.0,Delhi,Electric,"52,000 Kms",First,https://www.cartrade.com/second-hand/delhi/tat...,2021
2,2023 Toyota Innova Hycross ZX (O) Hybrid (Elec...,2950000.0,Delhi,Hybrid (Electric + Petrol),"56,835 Kms",First,https://www.cartrade.com/second-hand/delhi/toy...,2023
3,2021 Toyota Camry Hybrid,2690000.0,Delhi,Hybrid (Electric + Petrol),"63,000 Kms",First,https://www.cartrade.com/second-hand/delhi/toy...,2021
4,2019 Mercedes-Benz C-Class C 200 Progressive [...,3149000.0,Delhi,Petrol,"40,500 Kms",First,https://www.cartrade.com/second-hand/delhi/mer...,2019
5,2024 Maruti Suzuki Vitara Brezza Zxi Plus Petr...,1125000.0,Delhi,Petrol,"13,000 Kms",First,https://www.cartrade.com/second-hand/delhi/mar...,2024


In [17]:
df["title_clean"] = df["title"].str.replace(r'^\d{4}\s+', '', regex=True)

In [18]:
df["brand"] = df["title_clean"].str.split().str[0]

In [19]:
df["model"] = df["title_clean"].str.split().str[1]

In [20]:
df = df.drop(columns=["title_clean"])

In [21]:
df.head()

,title,price,city,fuel,kms,owners,url,year,brand,model
1,2021 Tata Nexon EV XZ Plus,860000.0,Delhi,Electric,"52,000 Kms",First,https://www.cartrade.com/second-hand/delhi/tat...,2021,Tata,Nexon
2,2023 Toyota Innova Hycross ZX (O) Hybrid (Elec...,2950000.0,Delhi,Hybrid (Electric + Petrol),"56,835 Kms",First,https://www.cartrade.com/second-hand/delhi/toy...,2023,Toyota,Innova
3,2021 Toyota Camry Hybrid,2690000.0,Delhi,Hybrid (Electric + Petrol),"63,000 Kms",First,https://www.cartrade.com/second-hand/delhi/toy...,2021,Toyota,Camry
4,2019 Mercedes-Benz C-Class C 200 Progressive [...,3149000.0,Delhi,Petrol,"40,500 Kms",First,https://www.cartrade.com/second-hand/delhi/mer...,2019,Mercedes-Benz,C-Class
5,2024 Maruti Suzuki Vitara Brezza Zxi Plus Petr...,1125000.0,Delhi,Petrol,"13,000 Kms",First,https://www.cartrade.com/second-hand/delhi/mar...,2024,Maruti,Suzuki


In [22]:
df["kms"] = df["kms"].str.replace(",", "")
df["kms"] = df["kms"].str.extract(r'(\d+)')
df["kms"] = df["kms"].astype(int)

In [24]:
brands = [
"Maruti Suzuki","Hyundai","Honda","Toyota","Tata","Mahindra",
"Mercedes-Benz","BMW","Audi","Kia","Volkswagen","Skoda",
"Renault","Nissan","Jeep","Volvo","MG","Land Rover","Jaguar"
]

In [25]:
def extract_brand_model(title):

    title = title.replace(str(title.split()[0]), "").strip()

    for brand in brands:
        if brand in title:
            model = title.replace(brand,"").strip()
            return brand, model

    parts = title.split()
    return parts[0], " ".join(parts[1:])

In [26]:
df[["brand","model"]] = df["title"].apply(
    lambda x: pd.Series(extract_brand_model(x))
)

In [34]:
df.to_excel('df.xlsx')

In [35]:
df.head()

,title,price,city,fuel,kms,owners,url,year,brand,model
1,2021 Tata Nexon EV XZ Plus,860000.0,Delhi,Electric,52000,First,https://www.cartrade.com/second-hand/delhi/tat...,2021,Tata,Nexon EV XZ Plus
2,2023 Toyota Innova Hycross ZX (O) Hybrid (Elec...,2950000.0,Delhi,Hybrid (Electric + Petrol),56835,First,https://www.cartrade.com/second-hand/delhi/toy...,2023,Toyota,Innova Hycross ZX (O) Hybrid (Electric + Petro...
3,2021 Toyota Camry Hybrid,2690000.0,Delhi,Hybrid (Electric + Petrol),63000,First,https://www.cartrade.com/second-hand/delhi/toy...,2021,Toyota,Camry Hybrid
4,2019 Mercedes-Benz C-Class C 200 Progressive [...,3149000.0,Delhi,Petrol,40500,First,https://www.cartrade.com/second-hand/delhi/mer...,2019,Mercedes-Benz,C-Class C 200 Progressive [2018-2020]
5,2024 Maruti Suzuki Vitara Brezza Zxi Plus Petr...,1125000.0,Delhi,Petrol,13000,First,https://www.cartrade.com/second-hand/delhi/mar...,2024,Maruti Suzuki,Vitara Brezza Zxi Plus Petrol Manual


In [38]:
models = [
"Swift","Brezza","Vitara","Baleno","Alto","WagonR",
"Nexon","Harrier","Punch",
"XUV300","XUV500","XUV700","Scorpio",
"Fortuner","Innova","Camry",
"Creta","Venue","i20",
"City","Amaze",
"C-Class","E-Class","GLA","GLC"
]

In [39]:
def clean_model(name):

    for m in models:
        if m.lower() in name.lower():
            return m

    return name.split()[0]

In [40]:
def clean_model(name):

    for m in models:
        if m.lower() in name.lower():
            return m

    return name.split()[0]

In [41]:
df["model_clean"] = df["model"].apply(clean_model)

In [42]:
df["model_clean"] = df["model"].apply(clean_model)

In [43]:
owner_map = {"First":1,"Second":2,"Third":3,"Fourth":4}

df["owners"] = df["owners"].map(owner_map)

In [44]:
df.head()

,title,price,city,fuel,kms,owners,url,year,brand,model,model_clean
1,2021 Tata Nexon EV XZ Plus,860000.0,Delhi,Electric,52000,1.0,https://www.cartrade.com/second-hand/delhi/tat...,2021,Tata,Nexon EV XZ Plus,Nexon
2,2023 Toyota Innova Hycross ZX (O) Hybrid (Elec...,2950000.0,Delhi,Hybrid (Electric + Petrol),56835,1.0,https://www.cartrade.com/second-hand/delhi/toy...,2023,Toyota,Innova Hycross ZX (O) Hybrid (Electric + Petro...,Innova
3,2021 Toyota Camry Hybrid,2690000.0,Delhi,Hybrid (Electric + Petrol),63000,1.0,https://www.cartrade.com/second-hand/delhi/toy...,2021,Toyota,Camry Hybrid,Camry
4,2019 Mercedes-Benz C-Class C 200 Progressive [...,3149000.0,Delhi,Petrol,40500,1.0,https://www.cartrade.com/second-hand/delhi/mer...,2019,Mercedes-Benz,C-Class C 200 Progressive [2018-2020],C-Class
5,2024 Maruti Suzuki Vitara Brezza Zxi Plus Petr...,1125000.0,Delhi,Petrol,13000,1.0,https://www.cartrade.com/second-hand/delhi/mar...,2024,Maruti Suzuki,Vitara Brezza Zxi Plus Petrol Manual,Brezza


In [45]:
df.to_csv('output.csv', index=False, encoding='utf-8')